In [14]:
import pandas as pd


Convert all .csv files in pmsys folders into .json files, reformatting the first column as a datetime and renaming files when appropriate (like srpe.csv → session_rating_of_perceived_exertion.json).

In [125]:
import pandas as pd
from pathlib import Path

pmdata = Path('/Users/shreyavora/Desktop/HealTheTileLLM/pmdata')

# Iterate through only files in pmsys folders
for pmsys_dir in pmdata.rglob('pmsys'):
    if pmsys_dir.is_dir():
        for file_path in pmsys_dir.rglob('*'):
            if file_path.is_file() and file_path.suffix == '.csv':
                # Read file
                df = pd.read_csv(file_path)

                # Reformat first column
                df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0]).dt.strftime('%Y-%m-%d %H:%M:%S')
                df.rename(columns={df.columns[0]: 'dateTime'}, inplace=True)

                # Convert CSV to JSON
                json_file_path = file_path.with_suffix('.json')
                df.to_json(json_file_path, orient='records', date_format='iso')

                # Rename the JSON file if original was srpe.csv
                if file_path.name == 'srpe.csv':
                    renamed_json_path = file_path.with_name('session_rating_of_perceived_exertion.json')
                    json_file_path.rename(renamed_json_path)


Convert every fitbit/sleep_score.csv file inside the pmdata folder (and its subfolders) into a properly formatted sleep_score.json file.

In [126]:
from pathlib import Path
import pandas as pd

pmdata = Path('/Users/shreyavora/Desktop/HealTheTileLLM/pmdata')

for file_path in pmdata.rglob("fitbit/sleep_score.csv"):
    df = pd.read_csv(file_path)
    df.rename(columns={df.columns[0]: 'dateTime'}, inplace=True)
    df['dateTime'] = pd.to_datetime(df['dateTime']).dt.strftime('%Y-%m-%dT%H:%M:%S')
    df.to_json(file_path.with_suffix(".json"), orient="records", indent=2)


This script recursively searches all folders under pmdata for fitbit/exercise.json and fitbit/sleep_score.json files. It updates each file by renaming timestamp-related columns (startTime → dateTime for exercise, and timestamp → dateTime for sleep_score), standardizing the format to YYYY-MM-DDTHH:MM:SS and removing any timezone info from sleep score timestamps.

In [127]:
import json
import pandas as pd
from pathlib import Path

def rename_exercise_file(file_path, overwrite=True):
    with open(file_path, "r") as file:
        data = json.load(file)
    df = pd.DataFrame(data)
    df = df.rename(columns={"startTime": "dateTime"})
    df.to_json(file_path, orient='records', indent=2)

# Set root directory
pmdata = Path('/Users/shreyavora/Desktop/HealTheTileLLM/pmdata')

# Search all `exercise.json` files under each PXX/fitbit/ folder
for exercise_file in pmdata.rglob("fitbit/exercise.json"):
    rename_exercise_file(exercise_file, overwrite=True)

def rename_sleep_score_file(file_path, overwrite=True):
    with open(file_path, "r") as file:
        data = json.load(file)
    df = pd.DataFrame(data)
    df = df.rename(columns={"timestamp": "dateTime"})
    df['dateTime'] = pd.to_datetime(df['dateTime']).dt.tz_localize(None)
    df["dateTime"] = df["dateTime"].dt.strftime("%Y-%m-%dT%H:%M:%S")  # Strip timezone
    df.to_json(file_path, orient='records', indent=2)

for sleep_file in pmdata.rglob("fitbit/sleep_score.json"):
    rename_sleep_score_file(sleep_file, overwrite=True)



In [32]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p01/pmsys/wellness.json")

In [17]:
def normalize_sleep_datetime(file_path, overwrite=True):
    with open(file_path, "r") as file:
        data = json.load(file)

    df = pd.DataFrame(data)

    if "dateTime" not in df.columns:
        raise ValueError("Missing 'dateTime' in input data.")

    # Step 1: Parse and normalize, coerce errors to NaT
    df["dateTime"] = pd.to_datetime(df["dateTime"], errors="coerce").dt.normalize()

    # Step 2: Remove rows with invalid dateTime (NaT)
    df = df.dropna(subset=["dateTime"])

    # Step 3: Format clean dateTime values
    df["dateTime"] = df["dateTime"].dt.strftime("%Y-%m-%dT%H:%M:%S")

    # Step 4: Save
    output_path = file_path if overwrite else Path(file_path).with_name("sleep_normalized.json")
    df.to_json(output_path, orient="records", indent=2, date_format="iso")


In [33]:
#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

In [34]:
# 1. Create the "exercise" column as a full nested dict per row
exercise_copy = exercise_data.copy()

nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})



In [35]:
merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


In [36]:
import pandas as pd
import json

dataP01 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # 🛡 SAFER datetime handling
    dt = row["dateTime"]

    if pd.isna(dt):
        continue
    elif isinstance(dt, pd.Timestamp):
        date_str = dt.isoformat()
    else:
        date_str = str(dt)

    dataP01.append({
        "dateTime": date_str,
        "healthDomain": health_domain
    })

# Save to JSON
with open("P01Data.json", "w") as f:
    json.dump(dataP01, f, indent=2)


In [22]:
import pandas as pd
import json

dataP01 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP01.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P01Data.json", "w") as f:
    json.dump(dataP01, f, indent=2)


In [155]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p02/pmsys/wellness.json")

In [156]:
#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

In [157]:
# 1. Create the "exercise" column as a full nested dict per row
exercise_copy = exercise_data.copy()

nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})



In [158]:
merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


In [159]:
import pandas as pd
import json

dataP02 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP02.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P02Data.json", "w") as f:
    json.dump(dataP02, f, indent=2)


In [160]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p03/pmsys/wellness.json")

In [161]:
#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

In [162]:
# 1. Create the "exercise" column as a full nested dict per row
exercise_copy = exercise_data.copy()

nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})



In [163]:
merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


In [164]:
import pandas as pd
import json

dataP03 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP03.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P03Data.json", "w") as f:
    json.dump(dataP03, f, indent=2)


In [165]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p04/pmsys/wellness.json")

In [166]:
#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

# 1. Create the "exercise" column as a full nested dict per row
exercise_copy = exercise_data.copy()

nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


In [167]:
import pandas as pd
import json

dataP04 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP04.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P04Data.json", "w") as f:
    json.dump(dataP04, f, indent=2)


In [168]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p05/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

# 1. Create the "exercise" column as a full nested dict per row
exercise_copy = exercise_data.copy()

nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


dataP05 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP05.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P05Data.json", "w") as f:
    json.dump(dataP05, f, indent=2)





In [169]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p06/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

# 1. Create the "exercise" column as a full nested dict per row
exercise_copy = exercise_data.copy()

nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


dataP06 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP06.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P06Data.json", "w") as f:
    json.dump(dataP06, f, indent=2)





In [170]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p07/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

# 1. Create the "exercise" column as a full nested dict per row
exercise_copy = exercise_data.copy()

nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


dataP07 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP07.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P07Data.json", "w") as f:
    json.dump(dataP07, f, indent=2)





In [172]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/fitbit/very_active_minutes.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p08/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

# 1. Create the "exercise" column as a full nested dict per row
exercise_copy = exercise_data.copy()

nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, srpe, on='dateTime', how='outer')
merged14 = pd.merge(merged13, wellness, on='dateTime', how='outer')


dataP08 = []

for _, row in merged14.iterrows():
    health_domain = {}

    for col in merged14.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP08.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P08Data.json", "w") as f:
    json.dump(dataP08, f, indent=2)





In [173]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p09/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

exercise_copy = exercise_data.copy()
nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


dataP09 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP09.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P09Data.json", "w") as f:
    json.dump(dataP09, f, indent=2)





In [174]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p10/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

exercise_copy = exercise_data.copy()
nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


dataP10 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP10.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P10Data.json", "w") as f:
    json.dump(dataP10, f, indent=2)





In [175]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p11/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

exercise_copy = exercise_data.copy()
nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


dataP11 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP11.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P11Data.json", "w") as f:
    json.dump(dataP11, f, indent=2)





In [176]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/heart_rate.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/moderately_active_minutes.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p12/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

exercise_copy = exercise_data.copy()
nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})




sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, moderately_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, sedentary_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, sleep_score, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sleep_data, on='dateTime', how='outer')
merged8 = pd.merge(merged7, steps, on='dateTime', how='outer')
merged9 = pd.merge(merged8, time_in_heart_zone, on='dateTime', how='outer')
merged10 = pd.merge(merged9, very_active_minutes, on='dateTime', how='outer')
merged11 = pd.merge(merged10, injury, on='dateTime', how='outer')
merged12 = pd.merge(merged11, srpe, on='dateTime', how='outer')
merged13 = pd.merge(merged12, wellness, on='dateTime', how='outer')


dataP12 = []

for _, row in merged13.iterrows():
    health_domain = {}

    for col in merged13.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP12.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P12Data.json", "w") as f:
    json.dump(dataP12, f, indent=2)





In [178]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/moderately_active_minutes.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p13/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

exercise_copy = exercise_data.copy()
nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})




sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, sedentary_minutes, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sleep_score, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_data, on='dateTime', how='outer')
merged9 = pd.merge(merged8, steps, on='dateTime', how='outer')
merged10 = pd.merge(merged9, time_in_heart_zone, on='dateTime', how='outer')
merged11 = pd.merge(merged10, very_active_minutes, on='dateTime', how='outer')
merged12 = pd.merge(merged11, injury, on='dateTime', how='outer')
merged13 = pd.merge(merged12, srpe, on='dateTime', how='outer')
merged14 = pd.merge(merged13, wellness, on='dateTime', how='outer')


dataP13 = []

for _, row in merged14.iterrows():
    health_domain = {}

    for col in merged14.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP13.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P13Data.json", "w") as f:
    json.dump(dataP13, f, indent=2)





In [179]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p14/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

exercise_copy = exercise_data.copy()
nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


dataP14 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP14.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P14Data.json", "w") as f:
    json.dump(dataP14, f, indent=2)





In [5]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p15/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

exercise_copy = exercise_data.copy()
nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


dataP15 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP15.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

# Save to JSON
with open("P15Data.json", "w") as f:
    json.dump(dataP15, f, indent=2)





In [6]:
calories_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/calories.json")  # Gene expression data
distance_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/distance.json")
exercise_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/exercise.json")
heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/heart_rate.json")
lightly_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/lightly_active_minutes.json")
moderately_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/moderately_active_minutes.json")
resting_heart_rate = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/resting_heart_rate.json")
sedentary_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/sedentary_minutes.json")
sleep_score = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/sleep_score.json")
sleep_data = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/sleep.json")
steps = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/steps.json")
time_in_heart_zone = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/time_in_heart_rate_zones.json")
very_active_minutes = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/fitbit/very_active_minutes.json")
injury = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/pmsys/injury.json")
srpe = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/pmsys/session_rating_of_perceived_exertion.json")
wellness = pd.read_json("/Users/shreyavora/Desktop/HealTheTileLLM/pmdata/p16/pmsys/wellness.json")



#all columns that need to be renamed
calories_data = calories_data.rename(columns= {"value" : "calories"})
distance_data = distance_data.rename(columns= {"value" : "distance"})
heart_rate = heart_rate.rename(columns= {"value" : "heart_rate"})
lightly_active_minutes = lightly_active_minutes.rename(columns= {"value" : "lightly_active_minutes"})
moderately_active_minutes = moderately_active_minutes.rename(columns= {"value" : "moderately_active_minutes"})
resting_heart_rate = resting_heart_rate.rename(columns= {"value" : "resting_heart_rate"})
sedentary_minutes = sedentary_minutes.rename(columns= {"value" : "sedentary_minutes"})
steps = steps.rename(columns= {"value" : "steps"})
time_in_heart_zone = time_in_heart_zone.rename(columns= {"value" : "time_in_heart_rate_zone"})
very_active_minutes = very_active_minutes.rename(columns= {"value" : "very_active_minutes"})

exercise_copy = exercise_data.copy()
nested = exercise_copy.drop(columns=["dateTime"]).to_dict(orient="records")
exercise_data = pd.DataFrame({
    "dateTime": exercise_copy["dateTime"],
    "exercise": nested
})

resting_heart_rate_copy = resting_heart_rate.copy()
nested = resting_heart_rate_copy.drop(columns=["dateTime"]).to_dict(orient="records")
resting_heart_rate = pd.DataFrame({
    "dateTime": resting_heart_rate_copy["dateTime"],
    "resting_heart_rate": nested
})



sleep_score_copy = sleep_score.copy()
nested = sleep_score_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_score = pd.DataFrame({
    "dateTime": sleep_score_copy["dateTime"],
    "sleep_score": nested
})


sleep_copy = sleep_data.copy()
nested = sleep_copy.drop(columns=["dateTime"]).to_dict(orient="records")
sleep_data = pd.DataFrame({
    "dateTime": sleep_copy["dateTime"],
    "sleep": nested
})

time_in_heart_zone_copy = time_in_heart_zone.copy()
nested = time_in_heart_zone_copy.drop(columns=["dateTime"]).to_dict(orient="records")
time_in_heart_zone = pd.DataFrame({
    "dateTime": time_in_heart_zone_copy["dateTime"],
    "time_in_heart_zone": nested
})

srpe_copy = srpe.copy()
nested = srpe_copy.drop(columns=["dateTime"]).to_dict(orient="records")
srpe = pd.DataFrame({
    "dateTime": srpe_copy["dateTime"],
    "session_rating_of_perceived_exertion": nested
})

wellness_copy = wellness.copy()
nested = wellness_copy.drop(columns=["dateTime"]).to_dict(orient="records")
wellness = pd.DataFrame({
    "dateTime": wellness_copy["dateTime"],
    "wellness": nested
})

merged = pd.merge(distance_data, calories_data, on='dateTime', how='outer')
merged2 = pd.merge(merged, exercise_data, on='dateTime', how='outer')
merged3 = pd.merge(merged2, heart_rate, on='dateTime', how='outer')
merged4 = pd.merge(merged3, lightly_active_minutes, on='dateTime', how='outer')
merged5 = pd.merge(merged4, moderately_active_minutes, on='dateTime', how='outer')
merged6 = pd.merge(merged5, resting_heart_rate, on='dateTime', how='outer')
merged7 = pd.merge(merged6, sedentary_minutes, on='dateTime', how='outer')
merged8 = pd.merge(merged7, sleep_score, on='dateTime', how='outer')
merged9 = pd.merge(merged8, sleep_data, on='dateTime', how='outer')
merged10 = pd.merge(merged9, steps, on='dateTime', how='outer')
merged11 = pd.merge(merged10, time_in_heart_zone, on='dateTime', how='outer')
merged12 = pd.merge(merged11, very_active_minutes, on='dateTime', how='outer')
merged13 = pd.merge(merged12, injury, on='dateTime', how='outer')
merged14 = pd.merge(merged13, srpe, on='dateTime', how='outer')
merged15 = pd.merge(merged14, wellness, on='dateTime', how='outer')


dataP16 = []

for _, row in merged15.iterrows():
    health_domain = {}

    for col in merged15.columns:
        if col == "dateTime":
            continue

        value = row[col]

        # Use pandas isna to detect missing values robustly
        if not value:
            health_domain[col] = None
        else:
            health_domain[col] = value

    # Build the final structured object
    dataP16.append({
        "dateTime": row["dateTime"].isoformat() if isinstance(row["dateTime"], pd.Timestamp) else row["dateTime"],
        "healthDomain": health_domain
    })

c





In [1]:
import os
import shutil

# Define the source directory (current working directory)
source_dir = "/Users/shreyavora/Desktop/HealTheTileLLM"  # Replace with your actual path
target_dir = os.path.join(source_dir, "Integrated_Datasets")

# Create the target folder if it doesn't exist
os.makedirs(target_dir, exist_ok=True)

# Move all matching files
for filename in os.listdir(source_dir):
    if filename.startswith("P") and filename.endswith("Data.json"):
        src = os.path.join(source_dir, filename)
        dst = os.path.join(target_dir, filename)
        shutil.move(src, dst)
        print(f"Moved {filename} to Integrated_Datasets/")
